# 06 — Tagging production

**Purpose.** Run the frozen two-pass tagging instrument
(`item1-topic-tagging-1.0`) over every block 03 built: Pass A (label gate)
on all blocks, Pass B (definition confirm + direction) where the gate found
candidates. Sharded, resumable, every call logged.

**Run shape.** `jobs/tag_item1.sbatch`, array 0–9; block `i` belongs to
shard `i mod N_SHARDS`. A task that dies is re-submitted, not restarted:
finished blocks are read back from the shard parquet and skipped.

**Outputs**, per shard under `tagging/shards/`:

| File | Contents |
|---|---|
| `gate_shard{K}.parquet` | one row per (block, gated topic): topic_id, n_cited |
| `tags_shard{K}.parquet` | one row per confirmed tag: sent_idx, topic_id, direction |
| `blocks_shard{K}.parquet` | one row per processed block: counts, errors, seconds |
| `status/shard{K}.jsonl` | full CallLog (prompt hash, complete output, timing) |

**Monitoring bands** (checked in the verification cell and by eye on the
logs): gate parse-clean ≥ 95%; confirm parse-clean ≥ 95% (salvage counted
separately); share of blocks with ≥1 candidate — the dev read saw 3/8, so
0.2–0.6 is the plausible band; outside it, stop the array and read shards.

**Aggregation to (firm, fyear, topic) cells happens in 07** after all
shards land — related (≥1 tag; ≥2 as sensitivity), intensity = tag counts,
direction = tag shares, INV-25 hearing join.

## Setup — shard assignment and resume state

In [ ]:
import hashlib
import json
import os
import sys
import time
from collections import Counter

sys.path.insert(0, "../lib")

import pandas as pd
import torch

from paths import HF_CACHE, TOPICS_INTERIM, relevance_run_dir
import tagging as T
import summarize as S

os.environ["HF_HOME"] = str(HF_CACHE)

MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
SHARD = int(os.environ.get("SHARD", "0"))
N_SHARDS = int(os.environ.get("N_SHARDS", "10"))
FLUSH_EVERY = int(os.environ.get("FLUSH_EVERY", "200"))

assert not T.PROMPT_VERSION.endswith("-dev"), \
    "production refuses to run an unfrozen prompt"

ITEM1 = relevance_run_dir("item1")
TAG = relevance_run_dir("tagging")
SHARDS = TAG / "shards"
(SHARDS / "status").mkdir(parents=True, exist_ok=True)

GATE_FILE = SHARDS / f"gate_shard{SHARD}.parquet"
TAGS_FILE = SHARDS / f"tags_shard{SHARD}.parquet"
BLOCKS_FILE = SHARDS / f"blocks_shard{SHARD}.parquet"

codebook = pd.read_parquet(TOPICS_INTERIM / "taxonomy" / "codebook_v1.parquet")
LABELS_TEXT, TOPIC_IDS = T.format_topic_labels(codebook)

blocks = pd.read_parquet(TAG / "blocks.parquet").reset_index(drop=True)
work = blocks.loc[blocks.index % N_SHARDS == SHARD].reset_index(drop=True)

done = set()
if BLOCKS_FILE.exists():
    prev = pd.read_parquet(BLOCKS_FILE)
    done = set(map(tuple, prev[["accessionNo", "block_idx"]]
                   .itertuples(index=False)))
todo = work.loc[[tuple(x) not in done for x in
                 work[["accessionNo", "block_idx"]]
                 .itertuples(index=False)]].reset_index(drop=True)

sentences = pd.read_parquet(ITEM1 / "sentences.parquet",
                            columns=["accessionNo", "sent_idx", "sentence"])
sentences = sentences.loc[sentences["accessionNo"].isin(set(todo["accessionNo"]))]
sent_by_acc = dict(tuple(sentences.groupby("accessionNo")))

print(f"prompt   : {T.PROMPT_VERSION}   model {MODEL}")
print(f"shard    : {SHARD} of {N_SHARDS}")
print(f"blocks   : {len(work):,} assigned, {len(done):,} done, "
      f"{len(todo):,} to run")

## Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto",
                                             device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s")


def seed_for(acc, block_idx, stage):
    torch.manual_seed(int.from_bytes(
        hashlib.sha256(f"{acc}:{block_idx}:{stage}".encode()).digest()[:8],
        "little"))

## Production loop

Per block: gate; if candidates, one confirm call with their definitions.
Accumulated rows are flushed to the shard parquets every `FLUSH_EVERY`
blocks and at the end, so a killed task loses at most one flush window.

In [ ]:
log = S.CallLog(SHARDS / "status" / f"shard{SHARD}.jsonl",
                f"shard{SHARD}", MODEL, prompt_version=T.PROMPT_VERSION)

gate_rows, tag_rows, block_rows = [], [], []


def flush():
    for path, rows in ((GATE_FILE, gate_rows), (TAGS_FILE, tag_rows),
                       (BLOCKS_FILE, block_rows)):
        new = pd.DataFrame(rows)
        if path.exists():
            new = pd.concat([pd.read_parquet(path), new], ignore_index=True)
        if len(new):
            new.to_parquet(path, index=False)
    gate_rows.clear(); tag_rows.clear(); block_rows.clear()


started = time.time()
for n_done, row in enumerate(todo.itertuples(), 1):
    grp = sent_by_acc[row.accessionNo]
    sents = grp.loc[(grp["sent_idx"] >= row.sent_lo)
                    & (grp["sent_idx"] <= row.sent_hi), "sentence"].tolist()
    start = row.sent_lo + 1
    errs = Counter()
    t_block = time.time()

    user = T.render_gate_user(sents, start, LABELS_TEXT)
    seed_for(row.accessionNo, row.block_idx, "gate")
    t0 = time.time()
    out, _ = S._generate(model, tokenizer, T.SYSTEM, user, T.GATE_MAX_NEW_TOKENS)
    log.write("gate", 0, n_done, len(sents), user, out, tokenizer,
              time.time() - t0)
    topics_hit, e = T.parse_gate_topics(out, TOPIC_IDS)
    errs.update(e)
    for tid in topics_hit:
        gate_rows.append({"accessionNo": row.accessionNo,
                          "block_idx": row.block_idx, "topic_id": tid})

    n_tags = 0
    if topics_hit:
        candidates = sorted(topics_hit)
        entries, cand_ids = T.format_topic_entries(codebook, candidates)
        user = T.render_confirm_user(sents, start, entries)
        seed_for(row.accessionNo, row.block_idx, "confirm")
        t0 = time.time()
        out, _ = S._generate(model, tokenizer, T.SYSTEM, user,
                             T.CONFIRM_MAX_NEW_TOKENS)
        log.write("confirm", 0, n_done, len(sents), user, out, tokenizer,
                  time.time() - t0)
        tags, e = T.parse_topic_tags(out, cand_ids, start, len(sents))
        errs.update(e)
        n_tags = len(tags)
        for t in tags:
            tag_rows.append({
                "accessionNo": row.accessionNo, "block_idx": row.block_idx,
                "sent_idx": t["sentence"] - 1, "topic_id": t["topic"],
                "direction": t["direction"]})

    block_rows.append({
        "accessionNo": row.accessionNo, "block_idx": row.block_idx,
        "n_sentences": len(sents), "n_candidates": len(topics_hit),
        "n_tags": n_tags, "errors": json.dumps(dict(errs)),
        "seconds": round(time.time() - t_block, 2)})

    if n_done % FLUSH_EVERY == 0:
        flush()
        rate = n_done / ((time.time() - started) / 3600)
        print(f"  {n_done:,}/{len(todo):,} blocks  "
              f"{rate:,.0f}/h  eta {(len(todo) - n_done) / rate:.1f}h",
              flush=True)

flush()
print(f"\nshard {SHARD} complete: {len(todo):,} blocks in "
      f"{(time.time() - started) / 3600:.1f}h")

## Verification — this shard against the monitoring bands

In [ ]:
bl = pd.read_parquet(BLOCKS_FILE)
err_counts = Counter()
for e in bl["errors"]:
    err_counts.update(json.loads(e))

n = len(bl)
hard_fail = sum(v for k, v in err_counts.items() if k == "json_parse")
print(f"blocks processed : {n:,} of {len(work):,} assigned")
print(f"gated share      : {(bl['n_candidates'] > 0).mean():.1%} "
      f"(band 20-60%)")
print(f"tags             : {int(bl['n_tags'].sum()):,}")
print(f"parse errors     : {dict(err_counts) or 'none'} "
      f"(hard json_parse: {hard_fail})")
print(f"seconds per block: median {bl['seconds'].median():.1f}")
assert n == len(work), "shard incomplete — resubmit this array index"
print("\nDONE shard", SHARD)